<a href="https://colab.research.google.com/github/Naveen-salugu-github/BigData/blob/master/hanifirst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pyspark==3.5.4

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("retail_calendar_445_colab")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.4


In [3]:
# True = generate demo calendar in-memory (no AWS needed)
USE_MOCK_DATA = True

# If USE_MOCK_DATA is False, set path after uploading to Colab (Files sidebar)
UPLOADED_D_CAL_PATH = "/content/d_cal.parquet"  # or .csv

RUN_ID = "colab_demo"
SOURCE_VIEW = "d_cal"  # temp view name used in SQL below

In [4]:
import calendar as cal_mod
from datetime import date, timedelta

DOW = ["MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN"]
MTH_SHORT = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MTH_LONG = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]


def build_mock_d_cal(start: date = date(2001, 1, 1), end: date = date(2026, 12, 31)):
    """Minimal d_cal shaped like us_commercial_app_commons_prod.d_cal."""
    rows = []
    cal_id = 1
    d = start
    while d <= end:
        wk_start = d - timedelta(days=(d.weekday() + 2) % 7)
        wk_end = wk_start + timedelta(days=6)
        mth_start = date(d.year, d.month, 1)
        mth_end = date(d.year, d.month, cal_mod.monthrange(d.year, d.month)[1])
        rows.append(
            (
                cal_id,
                d,
                str(d),
                "Y",
                d.year,
                mth_start,
                mth_end,
                MTH_SHORT[d.month - 1],
                MTH_LONG[d.month - 1],
                wk_start,
                wk_end,
                DOW[d.weekday()],
            )
        )
        cal_id += 1
        d += timedelta(days=1)
    return rows


if USE_MOCK_DATA:
    schema = """
        cal_dt_id INT, cal_dt DATE, cal_dt_desc STRING, actv_ind STRING,
        cal_yr_num INT, mth_strt_dt DATE, mth_end_dt DATE,
        mth_shrt_nm STRING, mth_long_nm STRING,
        wk_strt_dt DATE, wk_end_dt DATE, dy_of_wk_shrt_nm STRING
    """
    d_cal = spark.createDataFrame(build_mock_d_cal(), schema=schema)
    print(f"Mock d_cal: {d_cal.count():,} days")
else:
    path = UPLOADED_D_CAL_PATH
    if path.endswith(".parquet"):
        d_cal = spark.read.parquet(path)
    elif path.endswith(".csv"):
        d_cal = spark.read.option("header", True).csv(path)
    else:
        raise ValueError("Upload .parquet or .csv and set UPLOADED_D_CAL_PATH")
    print(f"Loaded d_cal from {path}")

d_cal.createOrReplaceTempView(SOURCE_VIEW)
d_cal.filter("actv_ind = 'Y'").orderBy("cal_dt").show(5, truncate=False)

Mock d_cal: 9,496 days
+---------+----------+-----------+--------+----------+-----------+----------+-----------+-----------+----------+----------+----------------+
|cal_dt_id|cal_dt    |cal_dt_desc|actv_ind|cal_yr_num|mth_strt_dt|mth_end_dt|mth_shrt_nm|mth_long_nm|wk_strt_dt|wk_end_dt |dy_of_wk_shrt_nm|
+---------+----------+-----------+--------+----------+-----------+----------+-----------+-----------+----------+----------+----------------+
|1        |2001-01-01|2001-01-01 |Y       |2001      |2001-01-01 |2001-01-31|Jan        |January    |2000-12-30|2001-01-05|MON             |
|2        |2001-01-02|2001-01-02 |Y       |2001      |2001-01-01 |2001-01-31|Jan        |January    |2000-12-30|2001-01-05|TUE             |
|3        |2001-01-03|2001-01-03 |Y       |2001      |2001-01-01 |2001-01-31|Jan        |January    |2000-12-30|2001-01-05|WED             |
|4        |2001-01-04|2001-01-04 |Y       |2001      |2001-01-01 |2001-01-31|Jan        |January    |2000-12-30|2001-01-05|THU     

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_TABLE = SOURCE_VIEW
TARGET_VIEW = "r_calendar_445"

anchor_dt = spark.sql(f"""
    SELECT MIN(cal_dt) AS anchor_dt
    FROM {SOURCE_TABLE}
    WHERE UPPER(dy_of_wk_shrt_nm) = 'SAT'
""").first()[0]

end_year = spark.sql(f"""
    SELECT CAST(MAX(cal_yr_num) AS INT) AS end_yr
    FROM {SOURCE_TABLE}
""").first()[0]

anchors = [(anchor_dt.year, anchor_dt)]
while anchors[-1][0] <= end_year:
    yr, strt = anchors[-1]
    candidate = strt + timedelta(days=364)
    next_strt = strt + timedelta(days=371 if candidate.year == yr else 364)
    anchors.append((yr + 1, next_strt))

year_rows = [
    (yr, strt, anchors[i + 1][1] - timedelta(days=1),
     (anchors[i + 1][1] - strt).days // 7)
    for i, (yr, strt) in enumerate(anchors[:-1])
]
retail_years = spark.createDataFrame(
    year_rows,
    schema="rtl_yr INT, rtl_yr_strt_dt DATE, rtl_yr_end_dt DATE, weeks_in_yr INT",
)

print("First anchor:", anchor_dt, "| end_year:", end_year)
retail_years.orderBy("rtl_yr").show(10)

First anchor: 2001-01-06 | end_year: 2026
+------+--------------+-------------+-----------+
|rtl_yr|rtl_yr_strt_dt|rtl_yr_end_dt|weeks_in_yr|
+------+--------------+-------------+-----------+
|  2001|    2001-01-06|   2002-01-04|         52|
|  2002|    2002-01-05|   2003-01-03|         52|
|  2003|    2003-01-04|   2004-01-02|         52|
|  2004|    2004-01-03|   2004-12-31|         52|
|  2005|    2005-01-01|   2006-01-06|         53|
|  2006|    2006-01-07|   2007-01-05|         52|
|  2007|    2007-01-06|   2008-01-04|         52|
|  2008|    2008-01-05|   2009-01-02|         52|
|  2009|    2009-01-03|   2010-01-01|         52|
|  2010|    2010-01-02|   2010-12-31|         52|
+------+--------------+-------------+-----------+
only showing top 10 rows



In [6]:
d_cal = spark.table(SOURCE_TABLE).filter(F.col("actv_ind") == "Y")

cal_assigned = d_cal.alias("c").join(
    retail_years.alias("ry"),
    (F.col("c.cal_dt") >= F.col("ry.rtl_yr_strt_dt"))
    & (F.col("c.cal_dt") <= F.col("ry.rtl_yr_end_dt")),
    "inner",
).select(
    "c.cal_dt_id",
    "c.cal_dt",
    "c.cal_dt_desc",
    "c.cal_yr_num",
    "c.mth_strt_dt",
    "c.mth_end_dt",
    "c.mth_shrt_nm",
    "c.mth_long_nm",
    "c.wk_strt_dt",
    "c.wk_end_dt",
    "ry.rtl_yr",
    "ry.weeks_in_yr",
    "ry.rtl_yr_strt_dt",
    ((F.datediff(F.col("c.cal_dt"), F.col("ry.rtl_yr_strt_dt")) / 7).cast("int") + 1).alias(
        "week_in_yr"
    ),
)

nov_cutoff = F.when(F.col("weeks_in_yr") == 53, F.lit(48)).otherwise(F.lit(47))

cal_445 = (
    cal_assigned.withColumn(
        "month_445_num",
        F.when(F.col("week_in_yr") <= 4, 1)
        .when(F.col("week_in_yr") <= 8, 2)
        .when(F.col("week_in_yr") <= 13, 3)
        .when(F.col("week_in_yr") <= 17, 4)
        .when(F.col("week_in_yr") <= 21, 5)
        .when(F.col("week_in_yr") <= 26, 6)
        .when(F.col("week_in_yr") <= 30, 7)
        .when(F.col("week_in_yr") <= 34, 8)
        .when(F.col("week_in_yr") <= 39, 9)
        .when(F.col("week_in_yr") <= 43, 10)
        .when(F.col("week_in_yr") <= nov_cutoff, 11)
        .otherwise(12),
    )
    .withColumn("week_445_start_date", F.expr("date_add(rtl_yr_strt_dt, (week_in_yr - 1) * 7)"))
    .withColumn("week_445_end_date", F.expr("date_add(rtl_yr_strt_dt, week_in_yr * 7 - 1)"))
)

month_445 = cal_445.groupBy("rtl_yr", "month_445_num").agg(
    F.min("cal_dt").alias("month_445_start_date"),
    F.max("cal_dt").alias("month_445_end_date"),
)

wom = (
    d_cal.select("mth_strt_dt", "wk_strt_dt")
    .distinct()
    .withColumn(
        "week_of_month",
        F.dense_rank().over(Window.partitionBy("mth_strt_dt").orderBy("wk_strt_dt")),
    )
)

SHORT = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
LONG = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

r_calendar_445 = (
    cal_445.alias("c")
    .join(
        month_445.alias("m"),
        (F.col("c.rtl_yr") == F.col("m.rtl_yr"))
        & (F.col("c.month_445_num") == F.col("m.month_445_num")),
        "left",
    )
    .join(
        wom.alias("wom"),
        (F.col("c.mth_strt_dt") == F.col("wom.mth_strt_dt"))
        & (F.col("c.wk_strt_dt") == F.col("wom.wk_strt_dt")),
        "left",
    )
    .select(
        F.col("c.cal_dt_id").alias("calendar_date_identifier"),
        F.col("c.cal_dt").alias("calendar_date"),
        F.col("c.cal_dt_desc").alias("calendar_date_description"),
        F.col("wom.week_of_month"),
        F.col("c.mth_shrt_nm").alias("month_calendar_short_name"),
        F.col("c.mth_long_nm").alias("month_calendar_long_name"),
        F.col("c.mth_strt_dt").alias("month_calendar_start_date"),
        F.col("c.mth_end_dt").alias("month_calendar_end_date"),
        F.col("c.cal_yr_num").alias("year_calendar_name"),
        F.element_at(F.array(*[F.lit(n) for n in SHORT]), F.col("c.month_445_num")).alias(
            "month_445_short_name"
        ),
        F.element_at(F.array(*[F.lit(n) for n in LONG]), F.col("c.month_445_num")).alias(
            "month_445_long_name"
        ),
        F.col("m.month_445_start_date"),
        F.col("m.month_445_end_date"),
        F.col("c.rtl_yr").alias("year_445_name"),
        F.col("c.week_445_start_date"),
        F.col("c.week_445_end_date"),
        F.col("c.week_in_yr").alias("week_in_445_year"),
        F.date_format(F.col("c.cal_dt"), "EEE").alias("day_of_week_short_name"),
        F.lit(RUN_ID).alias("run_id"),
        F.current_timestamp().alias("ins_dt"),
    )
    .distinct()
)

r_calendar_445.createOrReplaceTempView(TARGET_VIEW)
print("Rows:", r_calendar_445.count())

Rows: 9491


In [7]:
sample = (
    r_calendar_445.filter(F.col("year_445_name") == 2024)
    .filter(F.col("month_445_long_name").isin("January", "November", "December"))
    .orderBy("calendar_date")
    .select(
        "calendar_date",
        "day_of_week_short_name",
        "year_445_name",
        "month_445_long_name",
        "week_in_445_year",
        "week_445_start_date",
        "week_445_end_date",
    )
    .limit(30)
)
display(sample.toPandas())

,calendar_date,day_of_week_short_name,year_445_name,month_445_long_name,week_in_445_year,week_445_start_date,week_445_end_date
0,2024-01-06,Sat,2024,January,1,2024-01-06,2024-01-12
1,2024-01-07,Sun,2024,January,1,2024-01-06,2024-01-12
2,2024-01-08,Mon,2024,January,1,2024-01-06,2024-01-12
3,2024-01-09,Tue,2024,January,1,2024-01-06,2024-01-12
4,2024-01-10,Wed,2024,January,1,2024-01-06,2024-01-12
5,2024-01-11,Thu,2024,January,1,2024-01-06,2024-01-12
6,2024-01-12,Fri,2024,January,1,2024-01-06,2024-01-12
7,2024-01-13,Sat,2024,January,2,2024-01-13,2024-01-19
8,2024-01-14,Sun,2024,January,2,2024-01-13,2024-01-19
9,2024-01-15,Mon,2024,January,2,2024-01-13,2024-01-19


In [8]:
spark.sql(f"""
WITH per_month AS (
    SELECT year_445_name, month_445_long_name, month_445_start_date, month_445_end_date,
           COUNT(DISTINCT calendar_date_identifier) AS day_ct,
           COUNT(DISTINCT week_445_start_date) AS wk_ct
    FROM {TARGET_VIEW}
    GROUP BY 1,2,3,4
)
SELECT year_445_name, SUM(day_ct) AS total_days, SUM(wk_ct) AS total_weeks,
       COUNT(*) AS months_in_year
FROM per_month
GROUP BY year_445_name
ORDER BY year_445_name
""").show(30, truncate=False)

+-------------+----------+-----------+--------------+
|year_445_name|total_days|total_weeks|months_in_year|
+-------------+----------+-----------+--------------+
|2001         |364       |52         |12            |
|2002         |364       |52         |12            |
|2003         |364       |52         |12            |
|2004         |364       |52         |12            |
|2005         |371       |53         |12            |
|2006         |364       |52         |12            |
|2007         |364       |52         |12            |
|2008         |364       |52         |12            |
|2009         |364       |52         |12            |
|2010         |364       |52         |12            |
|2011         |371       |53         |12            |
|2012         |364       |52         |12            |
|2013         |364       |52         |12            |
|2014         |364       |52         |12            |
|2015         |364       |52         |12            |
|2016         |371       |53

In [9]:
spark.sql(f"""
SELECT date_format(week_445_start_date, 'EEE') AS week_start_dow,
       date_format(week_445_end_date, 'EEE') AS week_end_dow,
       COUNT(*) AS row_ct
FROM {TARGET_VIEW}
GROUP BY 1, 2
""").show()

+--------------+------------+------+
|week_start_dow|week_end_dow|row_ct|
+--------------+------------+------+
|           Sat|         Fri|  9491|
+--------------+------------+------+

